In [1]:
import os
import json
import sys
import trafilatura
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from part1.Util.scraper import fetch_website_links
from openai import OpenAI

In [2]:
load_dotenv(override=True)

MODEL = 'deepseek-chat'
deepseek = OpenAI(
    api_key=os.getenv('DEEPSEEK_API_KEY'),
    base_url="https://api.deepseek.com/v1"
)

Function for fetch news content

In [3]:
def fetch_website_contents(url):
    downloaded = trafilatura.fetch_url(url)
    if downloaded is None:
        return "No title found", ""

    result = trafilatura.extract(
        downloaded,
        output_format="json",
        favor_recall=True,
        include_links=False,
        include_images=False,
    )

    if result is None:
        return ""

    data = json.loads(result)
    content = data.get("text") or ""
    return content

In [8]:
# print(fetch_website_links("https://www.bbc.com/news"))

Build the user prompt that concatenate links

In [5]:
def construct_link_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for news article.
Pick ten links
Response with the URL in JSON format.
Do not include other links that are not considered in news article.

Links (some might be relative links):

"""

    links = fetch_website_links(url)
    user_prompt += "<links>\n"
    user_prompt += "\n".join(links)
    user_prompt += "\n</links>"
    return user_prompt

In [10]:
# print(construct_link_user_prompt("https://www.bbc.com/news"))

System prompt for find the relevant link

In [6]:
# Link system prompt

link_system_prompt = """
You are provided with a list of links found on a webpage.
You are going to select the links that relevant to news article.
You should not select the links that direct to a specific topic
You should response a list of links in JSON format as in the following example which include the title of the article and relate url link:

{
    "links": [
        {"url": "https://www.espn.com/mlb/story/_/id/48227697/venezuela-rallies-italy-make-first-wbc-final-vs-usa"}
    ]
}

"""

Function that calls LLM to find out relevant links

In [8]:
def select_relevant_links(url):
    response = deepseek.chat.completions.create(
        model='deepseek-reasoner',
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": construct_link_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

In [13]:
# select_relevant_links("https://www.bbc.com/news")

Fetch news articles and concatenate them

In [9]:
# Fetch news content
def fetch_news_articles(url):
    relevant_links = select_relevant_links(url)
    news_articles = "<news_articles>\n"
    for link in relevant_links['links']:
        news_articles += "\n<article_content>\n"
        news_articles += fetch_website_contents(link['url'])
        news_articles += f"\n Link: {link['url']}\n"
        news_articles += "\n</article_content>\n"
    news_articles += "</news_articles>"

    return news_articles

In [13]:
news_articles = fetch_news_articles("https://www.bbc.com/news")

Construct prompt for find out what the topics of the news articles are focused on

In [14]:
def construct_finding_news_topics_from_article_user_prompt(news_webside_name, url):
    user_prompt = f"Find out what the topics of the news articles are focused on from {news_webside_name}.\n\n"
    # user_prompt += fetch_news_articles(url)
    user_prompt += news_articles
    return user_prompt

Have a look of what are the topics we have for today's the news articles first

In [11]:
def topics_of_news_articles(news_webside_name, url):
    system_prompt = """
    You are an assistant that analyzes what the topics of the provided news articles are focused on.
    Response in Markdown format.
    """
    user_prompt = construct_finding_news_topics_from_article_user_prompt(news_webside_name, url)

    response = deepseek.chat.completions.create(
        model='deepseek-reasoner',
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content

In [12]:
display(Markdown(topics_of_news_articles("BBC News", "https://www.bbc.com/news")))

Based on the provided articles from BBC News, the primary topics are focused on the escalating conflict in Iran and its wide-ranging consequences, along with several other distinct regional and global issues. The analysis reveals a dominant theme of geopolitical instability and its ripple effects.

Here is a breakdown of the focused topics:

### 1. Geopolitical Conflict & Instability in the Middle East (Dominant Theme)
This is the central focus, with multiple articles detailing the war's causes, consequences, and global impact.
*   **Leadership Crisis in Iran:** The assassination of senior official Ali Larijani is analyzed as deepening a crisis within Iran's leadership, affecting its war strategy, domestic control, and nuclear negotiations.
*   **U.S. Domestic Politics & War Justification:** The resignation of a top U.S. counterterrorism official highlights internal criticism of the war, accusations of misinformation, and debates over the rationale for the conflict.
*   **Impact on Global Diplomacy:** The war causes the postponement of a high-stakes meeting between the U.S. and Chinese presidents, illustrating how the conflict is disrupting other major international relationships and agendas.
*   **Human Cost & Civilian Suffering:** Articles from Tehran and Kabul depict the war's brutal human impact:
    *   **Iran:** Describes a population living under "total repression" and constant fear of airstrikes, detailing the psychological toll and stifling of dissent.
    *   **Afghanistan/Pakistan Conflict:** Covers a deadly cross-border airstrike on a civilian drug rehab centre in Kabul, highlighting regional hostilities and tragic civilian casualties.

### 2. Regional & Domestic Issues Unrelated to the Iran Conflict
Several articles cover significant stories in other parts of the world.
*   **India's Pharmaceutical Industry:** A detailed analysis of how the expiry of a patent on weight-loss drugs (like Ozempic) in India could lead to a flood of cheaper generics, reshaping the global fight against obesity and diabetes.
*   **Australian Media & Legal Affairs:**
    *   **Media:** Reports on the high-profile sacking of a controversial radio host (Kyle Sandilands) and the cancellation of his top-rating show following an on-air dispute.
    *   **Legal:** Covers a court hearing for the alleged Bondi gunman, focusing on legal arguments about suppressing the identities of his family members who have received death threats.
*   **Scottish Social Policy:** Details the emotional debate and ultimate rejection of a bill to legalize assisted dying in Scotland, capturing the ethical and personal arguments on both sides.

### 3. Economic & Energy Security Ramifications
One article specifically analyzes the secondary economic effects of the Middle East conflict.
*   **India's Energy Security:** Examines the potential vulnerability of India's piped natural gas (PNG) supply due to the Iran war, as a significant portion of its liquefied natural gas (LNG) imports transit the disrupted Strait of Hormuz. It discusses government prioritization and potential price rises.

### Summary
The **overwhelming focus** of the provided news articles is on the **Iran war and its multifaceted consequences**, including regional leadership crises, U.S. political debates, disrupted global diplomacy, and severe humanitarian suffering. Alongside this dominant narrative, BBC News is also covering major national stories in **India, Australia, and Scotland** on topics ranging from health economics and media scandals to profound ethical legislation. The final article connects the geopolitical conflict directly to **global energy market anxieties**, particularly for major importers like India.

In [17]:
summarize_news_system_prompt = """
You are an assistant that analyzes the news articles from user input and create a humorous, entertaining summary for these news articles.
For each news article summary, include a title, a short summary and the link relate to the new article.
Response in Markdown format.
"""

In [16]:
def construct_news_summary_user_prompt(news_webside_name, url, topic):
    user_prompt = f"""
    You are provided a list of news articles from {news_webside_name}Read the news articles provide below, and create a summary for me to quick understand what is happening.
    Only summarize the topic relate to {topic}.\n\n

    """
    user_prompt += fetch_news_articles(url)
    return user_prompt

In [18]:
def create_news_summary(news_webside_name, url, topic):
    stream = deepseek.chat.completions.create(
        model='deepseek-reasoner',
        messages=[
            {"role": "system", "content": summarize_news_system_prompt},
            {"role": "user", "content": construct_news_summary_user_prompt(news_webside_name, url, topic)}
        ],
        stream=True
    )

    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [19]:
create_news_summary("BBC News", "https://www.bbc.com/news", "Geopolitical Conflict")

Here are the geopolitical conflict-related news articles from your list, served with a side of dark humor and a dash of international intrigue.

### 🎭 **Geopolitical Drama Digest: Bombs, Resignations, and Delayed Dinners**

#### 1. **Iran's Leadership Takes a Hit (Literally)**
**Title:** *"Game of Drones: The One Where Iran's Top Strategist Gets a Sudden Promotion to 'In Memoriam'"*
**Summary:** An Israeli airstrike removed Ali Larijani, one of Iran's most pivotal security chiefs, leaving the country's leadership looking more like a game of musical chairs where the music is missile sirens. With the war, domestic unrest, and nuclear talks now in the hands of an unknown successor, Iran's future looks about as stable as a Jenga tower in an earthquake zone.
**Link:** https://www.bbc.com/news/articles/cgqgxqekp89o

#### 2. **A Dramatic Exit in Washington**
**Title:** *"Not My War, Bro: Top US Counterterrorism Boss Quits, Throws Shade at Israel and 'The Lobby'"*
**Summary:** In a move fit for a political thriller, Donald Trump's counterterrorism director, Joe Kent, resigned spectacularly, accusing the administration of starting a war with Iran based on "misinformation" from Israel and its supporters. The White House called his claims laughable, while critics accused him of antisemitic tropes, proving that in D.C., even resignations come with a full-blown Twitter war.
**Link:** https://www.bbc.com/news/articles/cg4g66r3z40o

#### 3. **The Great Power Summit Gets "Rescheduled"**
**Title:** *"Sorry Xi, Can't Do Lunch: Trump Postpones China Trip Because War is a Full-Time Job"*
**Summary:** President Trump has delayed his high-stakes meeting with China's Xi Jinping, citing the need to babysit the Iran conflict. While China insists the delay has nothing to do with the Strait of Hormuz (wink, wink), the move highlights how the war has become the ultimate party-crasher for global diplomacy.
**Link:** https://www.bbc.com/news/articles/cn9e0z7v3nxo

#### 4. **A Truly Grim Ramadan in Kabul**
**Title:** *"Pakistani Airstrike Turns Kabul Rehab Centre into a Nightmare: 'It Was Like Doomsday'"*
**Summary:** A Pakistani airstrike hit a drug rehabilitation center in Kabul during Ramadan dinner, killing hundreds in a horrific escalation of cross-border hostilities. Pakistan insists it was targeting "terrorist infrastructure," not patients trying to get clean, adding a new, devastating chapter to the long-running feud between the neighbors.
**Link:** https://www.bbc.com/news/articles/ckg1kgz6wkgo

#### 5. **Life in Tehran: Dread, Drones, and Despair**
**Title:** *"Living Under the Boom: Iranians Juggle Fear of Bombs and Fear of the Secret Police"*
**Summary:** For citizens in Tehran, daily life is a nerve-wracking cocktail of listening for incoming drones and avoiding the ever-present security forces. The official propaganda touts martyrdom, but on the ground, people are just trying to survive a war they didn't start and a regime they increasingly fear.
**Link:** https://www.bbc.com/news/articles/cn9e0wrglgdo

#### 6. **India's Gas Supply: Waiting for the Other Shoe to Drop**
**Title:** *"Will the Kitchen Taps Run Dry? India Nervously Eyes Its Gas Pipelines as Hormuz Heats Up"*
**Summary:** The Iran war has already spooked India's LPG market, and now the country is sweating over its piped natural gas supply. While households are a priority, half of India's LNG imports sail through the troubled Strait of Hormuz, meaning the nation's energy security is currently on a tanker playing a high-stakes game of maritime dodgeball.
**Link:** https://www.bbc.com/news/articles/cj6dl175w01o